**Contributed workflow — live GeoLab validation is still required.**

Read [README.md](README.md) first. Use a dedicated database and a small date subset. These notebooks can modify database collections and write S3 objects. No CSS data or credentials are distributed here.

# Build index to s3 objects
s3 objects are equivalent to files.  They are just stored and managed in a different way than file systems.   This notebook demonstrates how to build an index of the holdings of data by Earthscope for the test TA data.   

Earthscope organizes their waveform in miniseed day volumes.   The day volumes are held in the s3 equivalent of a directory which they call a "prefix".   This notebook creates a MongoDB collection with the name "wf_s3" 
that will be used to drive processing to extract event data.   This notebook is indepent of that and more-or-less builds a directory of all file-like (s3 objects) linked to TA data for all of the year 2010.   Our test data does not cover the entire year, but we do the entire year to demonstrate that creating the index is very lightweight.  Fetching waveforms using it is not and is done in a different notebook,

In [ ]:
from mspasspy.client import Client
mspass_client=Client()
dask_client = mspass_client.get_scheduler()
db = mspass_client.get_database("ANF48")

This notebook will work one year at a time.  To run this for different years one need only change the year symbol in the next box.

In [ ]:
year=2014

In [ ]:
from earthscope_sdk import EarthScopeClient
from s3_worker_plugin import fetch_s3_client

# Keep the SDK client alive for the refresh callback throughout indexing.
client = EarthScopeClient()
session = client.user.get_boto3_session()
s3_client = fetch_s3_client(session)
BUCKET = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"


This module contains special functions used to run this workflow and the related load_waveform workflows. 

In [ ]:
import s3segment_reader as geos3

We need to get a list of networks that have data in the year we are processing.  The entire list is unnecessarily large and would create a bit inefficiency below.  There may be a more elegant way to do this with a mongodb aggregation but this is simpler with pymongo from my perspective.

In [ ]:
from obspy import UTCDateTime
starttime = UTCDateTime(year=year,month=1,day=1)
endtime = UTCDateTime(year=year+1,month=1,day=1)
query=dict()
query['$and'] = [
   {'Ptime' : {'$gte' : starttime.timestamp} },
   {'Ptime' : {'$lt' : endtime.timestamp} }
]
n=db.arrival_css30.count_documents(query)
print(f"Number of arrivals for {year}={n}")
net_set = set()
cursor=db.arrival_css30.find(query)
for doc in cursor:
    net=doc.get("net")
    if isinstance(net, str) and net:
        net_set.add(net)
print(f"Distinct networks in year {year}")
netlist=list()
for net in net_set:
    print(net)
    netlist.append(net)


In [ ]:
# Include neighboring days needed by padded windows at year boundaries.
from datetime import date, timedelta
first_day = date(year, 1, 1) - timedelta(days=1)
last_day = date(year + 1, 1, 1)
index_days = []
day = first_day
while day <= last_day:
    index_days.append((day.year, day.timetuple().tm_yday))
    day += timedelta(days=1)


Now use the functions we defined above to create a document for each s3 object with auxmd added.   Save the resuls to MongoDB with the collection name "wf_s3".   

In [ ]:
# always clear wf_s3 here to allow this to be run for a new year easily
db.drop_collection("wf_s3")

In [ ]:
import time 
t0=time.time()
prefix_list=list()
for net in netlist:
    for index_year, day in index_days:
        s3names,auxmd = geos3.fetch_net_day_list(s3_client,net,index_year,day,bucket=BUCKET)
        doclist=geos3.objlist2doclist(s3names,auxmd)
        if len(doclist)>0:
            db.wf_s3.insert_many(doclist)
        else:
            print(f"No data found in s3 for net={net} for day={day} of {year}")
t=time.time()
n=db.wf_s3.count_documents({})
print("Total number of documents now in wf_s3=",n)
print("Elapsed time to build index=",t-t0)

Note that process is fast enough there is no need to add the complexity of running it in parallel.   It looks like one could build an index for the entire TA data set in less than 30 minutes running serial.  

Before leaving, verify what a typical document looks like with the mspass pretty print function.

In [ ]:
from mspasspy.util.seismic import print_metadata
doc=db.wf_s3.find_one()
print_metadata(doc)

For this algorithm this index will speed processing enormously.

In [ ]:
from pymongo import ASCENDING
db.wf_s3.create_index([
       ("year",ASCENDING),
       ("jday",ASCENDING),
])

In [ ]:
# important summary to print 
ns3 = db.wf_s3.count_documents({})
nfailures = db.s3_read_failures.count_documents({})
print("Total number of wf_s3 documents created=",ns3)
print("Current size of s3_read_failures collection (should be zero)=",nfailures)

Comments relevant if you want to utilize this prototype for a larger data set assembly:
1.  The wf_s3 collection can get very large very fast.   Choose the bounds carefully.
2.  Creating wf_s3 is fast.  Don't create it until you need to for multiple reasons.
3.  When data set is assembled (your version of load_waveforms_s3 completes) you can run `db.drop_collection("wf_s3") to reduce storage.  wf_s3 is more-or-less a scratch collection.
4.  Don't run that last box until wf_s3 is fully populated.  Each index you add to a collection slows write speed because the index has to be updated after the document is saved. 

In [ ]:
# Close only after all indexing requests have finished.
s3_client.close()
client.close()
